In [71]:
import numpy as np
import xarray as xr
import xarray.testing as xrt
from sdm_eurec4a import RepositoryPath

from pathlib import Path

import matplotlib.pyplot as plt

In [33]:
clara = RepositoryPath(development_regime="levante_m300950")
nils = RepositoryPath(development_regime="levante_m301096") 

In [72]:
def get_clara_nils_dataset(relative_path : Path) -> [xr.Dataset, xr.Dataset] : 
    clara_ds = xr.open_dataset(clara.data_dir / relative_path)
    nils_ds = xr.open_dataset(nils.data_dir / relative_path)
    return dict(
        clara = clara_ds,
        nils = nils_ds
    )
    

# Cloud composite dataset

In [76]:

d = get_clara_nils_dataset(relative_path= Path("observation/cloud_composite/processed/cloud_composite_SI_units_20241025.nc"))
clara_cc = d['clara']
nils_cc = d["nils"]

xrt.assert_allclose(clara_cc, nils_cc, rtol=1e-20, atol=1e-10)
xrt.assert_identical(clara_cc, nils_cc)

AssertionError: Left and right Dataset objects are not identical
Differing coordinates:
L * radius                                     (radius) float64 1kB 1.25e-06 ...
    Differing variable attributes:
        Long_name: drop diameter
R * radius                                     (radius) float64 1kB 1.25e-06 ...
Differing data variables:
L   particle_size_distribution                 (radius, time) float64 322MB 0...
    Differing variable attributes:
        Long_name: Particle Size Distribution
R   particle_size_distribution                 (radius, time) float64 322MB 0...

# Compare the identified clusters datasets

In [77]:

d = get_clara_nils_dataset(relative_path= Path("observation/cloud_composite/processed/identified_clusters/identified_clusters_rain_mask_5.nc"))
clara_cc = d['clara']
nils_cc = d["nils"]

xrt.assert_allclose(clara_cc, nils_cc, rtol=1e-20, atol=1e-10)
xrt.assert_identical(clara_cc, nils_cc)

AssertionError: Left and right Dataset objects are not close
Differing data variables:
L   duration                     (time) timedelta64[ns] 5kB 00:00:05 ... 00:0...
R   duration                     (time) int64 5kB 5 21 18 1 1 ... 1 1 32 1 208

In [78]:
np.unique(clara_cc["duration"].dt.total_seconds() - nils_cc["duration"])

array([0.])

# Dropsonde dataset

In [79]:

d = get_clara_nils_dataset(relative_path= Path("observation/dropsonde/processed/drop_sondes.nc"))
clara_cc = d['clara']
nils_cc = d["nils"]

xrt.assert_allclose(clara_cc, nils_cc, rtol=1e-20, atol=1e-10)
xrt.assert_identical(clara_cc, nils_cc)

# Distance relation

In [80]:
d = get_clara_nils_dataset(relative_path= Path("observation/combined/distance/distance_dropsondes_identified_clusters_rain_mask_5.nc"))

clara_distance = d["clara"]
nils_distance = d["nils"]

xrt.assert_allclose(clara_distance, nils_distance, rtol=1e-20, atol=1e-10)
xrt.assert_identical(clara_distance, nils_distance)

AssertionError: Left and right Dataset objects are not close
Differing data variables:
L   temporal_distance          (time_identified_clouds, time_drop_sondes) timedelta64[ns] 5MB ...
R   temporal_distance          (time_identified_clouds, time_drop_sondes) int64 5MB ...

In [81]:
np.unique(clara_distance["temporal_distance"].dt.total_seconds() - nils_distance["temporal_distance"])

array([-2.34140425e+09, -2.34103712e+09, -2.34075590e+09, ...,
        2.05708186e+09,  2.05711333e+09,  2.05716028e+09], shape=(585318,))

# Input for CLEO

## Thermodynamics

In [86]:
d = get_clara_nils_dataset(relative_path=Path("model/input_v4.2/potential_temperature_parameters.nc"))

clara_pt = d["clara"]
nils_pt = d["nils"]

xrt.assert_allclose(clara_pt, nils_pt, rtol=1e-10, atol=1e3)
xrt.assert_identical(clara_pt, nils_pt)

AssertionError: Left and right Dataset objects are not identical
Differing data variables:
L   x_split   (cloud_id) float64 2kB 707.4 707.4 707.4 ... 824.7 811.2 789.8
R   x_split   (cloud_id) float64 2kB 707.4 707.4 707.4 ... 824.7 811.1 789.8
L   slope_2   (cloud_id) float64 2kB 0.004785 0.004785 ... 0.004364 0.004396
R   slope_2   (cloud_id) float64 2kB 0.004785 0.004785 ... 0.004364 0.004396
L   f_0       (cloud_id) float64 2kB 298.1 298.1 298.1 ... 298.2 298.2 298.0
R   f_0       (cloud_id) float64 2kB 298.1 298.1 298.1 ... 298.2 298.2 298.0

# Liquid water content

In [90]:
d = get_clara_nils_dataset(relative_path=Path("model/input_v4.2/particle_size_distribution_parameters.nc"))

nils_psd = d["nils"]
clara_psd = d["clara"]